# Buổi 2: Pipeline và kiến trúc mô hình dự báo tẩy trắng san hô

Notebook này được thiết kế cho một buổi giảng khoảng **90 phút**, dựa trên tài liệu `Bổ sung lý thuyết Buổi 2.docx`.

Trọng tâm của buổi học là **ý tưởng tiếp cận và kiến trúc mô hình**, không phải huấn luyện mô hình thật lâu. Các cell code chủ yếu dùng để quan sát dữ liệu, shape đầu vào, sơ đồ pipeline, skeleton kiến trúc và output của từng nhóm mô hình.

## Mục tiêu học tập

Sau buổi học, người học cần nắm được:

1. Cách biến dữ liệu quan sát theo ngày thành bài toán supervised learning `X -> y`.
2. Vì sao pipeline đúng quan trọng hơn việc chỉ chọn mô hình mạnh.
3. Cách lag, rolling và calendar feature tạo "trí nhớ" cho mô hình tabular.
4. Kiến trúc và ý tưởng của Decision Tree, Random Forest, XGBoost và LightGBM.
5. Cách tạo sequence window để đưa chuỗi thời gian vào LSTM, GRU, CNN-LSTM và attention/TFT-lite.
6. Ý tưởng ST-GNN khi dữ liệu vừa có thời gian vừa có không gian.
7. Vì sao bài toán này phù hợp với multi-task learning: một encoder, hai đầu ra.
8. Vì sao RMSE tốt chưa chắc event recall tốt trong bài toán cảnh báo.

## Kịch bản 90 phút

| Thời lượng | Nội dung | Trọng tâm giảng |
|---:|---|---|
| 0-8 phút | Từ quan sát sang `X -> y` | DHW là hồi quy, alert level là phân loại |
| 8-18 phút | Pipeline dự báo | Không leakage, horizon 28 ngày, split theo thời gian |
| 18-30 phút | Feature engineering | Lag, rolling, calendar feature là trí nhớ cho model bảng |
| 30-42 phút | Decision Tree và Random Forest | Ngưỡng, tương tác biến, bagging và voting/averaging |
| 42-52 phút | XGBoost và LightGBM | Boosting học bằng cách sửa lỗi, regularization, class weight |
| 52-60 phút | Vì sao deep learning không tự động tốt hơn | Tabular baseline vẫn rất mạnh nếu feature tốt |
| 60-70 phút | Sequence window | Input shape `samples x 90 days x features` |
| 70-82 phút | LSTM, GRU, CNN-LSTM, TFT-lite | Bộ nhớ, pattern cục bộ, attention, output heads |
| 82-87 phút | ST-GNN và multi-task | Node, edge, message passing, shared encoder |
| 87-90 phút | Đánh giá | RMSE, macro-F1, event recall và lớp hiếm |

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
TABULAR_PATH = PROCESSED / "tabular_supervised.csv"
AGGREGATE_PATH = PROCESSED / "aggregate_features.csv"
NODES_PATH = PROCESSED / "nodes.csv"
ADJACENCY_PATH = PROCESSED / "adjacency.npy"

tabular = pd.read_csv(TABULAR_PATH, parse_dates=["time", "target_time_h28"])
aggregate = pd.read_csv(AGGREGATE_PATH, parse_dates=["time", "target_time_h28"])

TARGET_REG = "target_CRW_DHW_h28"
TARGET_CLS = "target_alert_level_h28"
HORIZON_DAYS = 28
SEQUENCE_LENGTH = 90

print(f"Project root: {ROOT}")
print(f"Tabular supervised: {tabular.shape[0]:,} rows, {tabular.shape[1]:,} columns")
print(f"Aggregate time series: {aggregate.shape[0]:,} rows, {aggregate.shape[1]:,} columns")
print(f"Time range: {tabular['time'].min().date()} -> {tabular['time'].max().date()}")

## 1. Từ quan sát sang bài toán học máy

Dữ liệu ban đầu chỉ là các quan sát theo ngày: SST, SST anomaly, HotSpot, DHW, BAA. Mô hình không học từ "dữ liệu" một cách mơ hồ, mà học quan hệ:

```text
X = thông tin tại ngày t và quá khứ trước ngày t
y = trạng thái tại ngày t + 28
```

Bài toán có hai đầu ra:

- **Hồi quy**: dự báo `target_CRW_DHW_h28`, tức DHW sau 28 ngày.
- **Phân loại**: dự báo `target_alert_level_h28`, tức mức cảnh báo sau 28 ngày.

Điểm cần nhấn mạnh khi giảng: sai số hồi quy nhỏ gần ngưỡng cảnh báo vẫn có thể tạo lỗi phân loại lớn. Ví dụ DHW thật 4.1 nhưng dự báo 3.9 có thể làm mô hình bỏ sót Alert Level 1.

In [ ]:
sample_cols = [
    "time", "node_id", "CRW_SST", "CRW_HOTSPOT", "CRW_DHW", "alert_label",
    "target_time_h28", TARGET_REG, TARGET_CLS,
]
display(tabular[sample_cols].head(8))

alert_labels = {
    0: "No Stress", 1: "Watch", 2: "Warning", 3: "Alert Level 1",
    4: "Alert Level 2", 5: "Alert Level 3", 6: "Alert Level 4", 7: "Alert Level 5",
}
target_counts = (
    tabular[TARGET_CLS]
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("target_alert_level_h28")
    .reset_index(name="n")
)
target_counts["label"] = target_counts["target_alert_level_h28"].map(alert_labels)
target_counts["pct"] = 100 * target_counts["n"] / target_counts["n"].sum()
display(target_counts)

## 2. Pipeline quan trọng hơn việc chỉ chọn mô hình

Một pipeline đúng phải mô phỏng tình huống thật:

```text
quá khứ + hiện tại tại ngày t -> dự báo tương lai tại ngày t + 28
```

Ba nguyên tắc cần kiểm tra liên tục:

1. Không dùng thông tin sau ngày `t` để tạo feature cho ngày `t`.
2. Nhãn `h28` phải thật sự nằm ở ngày `t + 28`.
3. Train, validation và test phải chia theo thời gian, không shuffle ngẫu nhiên như dữ liệu bảng thông thường.

Trong buổi giảng, nên nhắc lại ví dụ "thi công bằng": mô hình chỉ được dùng những gì nó đáng lẽ biết tại thời điểm dự báo.

In [ ]:
from matplotlib.patches import Circle, FancyBboxPatch

ARCH_COLORS = {
    "data": "#e0f2fe",
    "feature": "#dcfce7",
    "encoder": "#fef3c7",
    "hidden": "#ede9fe",
    "head_reg": "#ffe4e6",
    "head_cls": "#fae8ff",
    "output": "#f1f5f9",
    "warning": "#fff1f2",
}
EDGE = "#334155"


def _setup_arch_canvas(title, subtitle=None, figsize=(14, 5.2)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.text(0.02, 0.965, title, fontsize=16, fontweight="bold", ha="left", va="top", color="#0f172a")
    if subtitle:
        ax.text(0.02, 0.91, subtitle, fontsize=10, ha="left", va="top", color="#475569")
    return fig, ax


def _draw_box(ax, x, y, w, h, title, subtitle="", face="#ffffff", edge=EDGE, lw=1.4, title_size=10.5, subtitle_size=8.5):
    box = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.018,rounding_size=0.025",
        linewidth=lw,
        edgecolor=edge,
        facecolor=face,
    )
    ax.add_patch(box)
    ax.text(x + w / 2, y + h * 0.62, title, ha="center", va="center", fontsize=title_size, fontweight="bold", color="#0f172a")
    if subtitle:
        ax.text(x + w / 2, y + h * 0.31, subtitle, ha="center", va="center", fontsize=subtitle_size, color="#475569", linespacing=1.15)
    return box


def _arrow(ax, start, end, color=EDGE, lw=1.8, rad=0.0):
    ax.annotate(
        "",
        xy=end,
        xytext=start,
        arrowprops=dict(
            arrowstyle="-|>",
            lw=lw,
            color=color,
            shrinkA=2,
            shrinkB=2,
            connectionstyle=f"arc3,rad={rad}",
        ),
    )


def _small_label(ax, x, y, text, color="#475569"):
    ax.text(x, y, text, fontsize=8.5, ha="center", va="center", color=color)


def draw_pipeline_architecture():
    fig, ax = _setup_arch_canvas(
        "Pipeline dự báo đúng cho horizon 28 ngày",
        "Mọi feature tại ngày t chỉ được dùng hiện tại và quá khứ; nhãn nằm ở t + 28.",
        figsize=(14, 4.8),
    )
    y = 0.52
    w, h = 0.15, 0.18
    steps = [
        (0.04, "Quan sát tại t", "SST, HotSpot\nDHW, BAA", ARCH_COLORS["data"]),
        (0.24, "Trí nhớ tabular", "lag + rolling\nkhông nhìn tương lai", ARCH_COLORS["feature"]),
        (0.44, "Mùa vụ", "sin/cos ngày\ntrong năm", ARCH_COLORS["feature"]),
        (0.64, "Model", "tree ensemble\nhoặc sequence net", ARCH_COLORS["encoder"]),
        (0.82, "Dự báo", "DHW + alert\nt + 28", ARCH_COLORS["output"]),
    ]
    for i, (x, title, subtitle, color) in enumerate(steps):
        _draw_box(ax, x, y, w, h, title, subtitle, face=color)
        if i < len(steps) - 1:
            _arrow(ax, (x + w, y + h / 2), (steps[i + 1][0], y + h / 2))

    ax.plot([0.035, 0.965], [0.40, 0.40], color="#94a3b8", lw=1.2, ls="--")
    _small_label(ax, 0.50, 0.365, "ranh giới thời điểm dự báo: mô hình không được biết dữ liệu sau ngày t")
    _draw_box(ax, 0.33, 0.14, 0.22, 0.13, "Feature bị cấm", "future rolling, target,\ndữ liệu sau ngày t", face=ARCH_COLORS["warning"], edge="#be123c", title_size=10)
    _draw_box(ax, 0.62, 0.14, 0.22, 0.13, "Nhãn hợp lệ", "target_CRW_DHW_h28\ntarget_alert_level_h28", face="#f8fafc", edge="#64748b", title_size=10)
    _arrow(ax, (0.73, 0.52), (0.73, 0.27), color="#64748b", rad=-0.08)
    plt.show()


def draw_ensemble_architecture():
    fig, axes = plt.subplots(2, 1, figsize=(14, 6.4))
    fig.suptitle("Kiến trúc ensemble tree: bagging khác boosting ở cách ghép cây", fontsize=16, fontweight="bold", y=0.98)
    panels = [
        (
            axes[0],
            "Random Forest: nhiều cây học song song",
            [
                {"title": "Feature table", "subtitle": "lag, rolling,\ncalendar", "color": ARCH_COLORS["data"]},
                {"title": "Bootstrap", "subtitle": "mỗi cây nhìn\nmẫu hơi khác", "color": ARCH_COLORS["feature"]},
                {"title": "Forest", "subtitle": "T1, T2, ... Tn\nrandom features", "color": ARCH_COLORS["encoder"]},
                {"title": "Aggregate", "subtitle": "mean cho DHW\nvote cho alert", "color": ARCH_COLORS["output"]},
            ],
        ),
        (
            axes[1],
            "Boosting: cây sau sửa lỗi cây trước",
            [
                {"title": "Dự báo thô", "subtitle": "base score", "color": "#f8fafc"},
                {"title": "Loss/residual", "subtitle": "sai ở đâu?", "color": ARCH_COLORS["warning"]},
                {"title": "Tree k", "subtitle": "học phần lỗi\ncòn lại", "color": ARCH_COLORS["encoder"]},
                {"title": "Update", "subtitle": "cộng với\nlearning rate", "color": ARCH_COLORS["feature"]},
                {"title": "Final model", "subtitle": "tổng nhiều cây\nđã regularize", "color": ARCH_COLORS["output"]},
            ],
        ),
    ]
    for ax, title, blocks in panels:
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.axis("off")
        ax.text(0.02, 0.88, title, fontsize=12.5, fontweight="bold", ha="left", color="#0f172a")
        n = len(blocks)
        left, right, gap = 0.04, 0.96, 0.035
        w = (right - left - gap * (n - 1)) / n
        y, h = 0.34, 0.28
        for i, block in enumerate(blocks):
            x = left + i * (w + gap)
            _draw_box(ax, x, y, w, h, block["title"], block["subtitle"], face=block["color"], title_size=10)
            if i < n - 1:
                _arrow(ax, (x + w, y + h / 2), (x + w + gap, y + h / 2))
    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()


def draw_multitask_architecture(title, encoder_blocks, note, input_subtitle="90 ngày x F feature", figsize=(14, 5.3)):
    fig, ax = _setup_arch_canvas(title, note, figsize=figsize)
    _draw_box(ax, 0.04, 0.43, 0.15, 0.19, "Input", input_subtitle, face=ARCH_COLORS["data"])
    block_x, block_w, block_h = 0.30, 0.28, 0.115
    ys = np.linspace(0.70, 0.24, len(encoder_blocks))
    previous_center = (0.19, 0.525)
    for i, (block_title, block_subtitle, color) in enumerate(encoder_blocks):
        y = float(ys[i])
        _draw_box(ax, block_x, y, block_w, block_h, block_title, block_subtitle, face=color, title_size=10)
        if i == 0:
            _arrow(ax, previous_center, (block_x, y + block_h / 2))
        else:
            _arrow(ax, (block_x + block_w / 2, float(ys[i - 1])), (block_x + block_w / 2, y + block_h))
    last_y = float(ys[-1])
    _draw_box(ax, 0.64, 0.43, 0.13, 0.19, "Hidden", "biểu diễn\nstress nhiệt", face=ARCH_COLORS["hidden"])
    _arrow(ax, (block_x + block_w, last_y + block_h / 2), (0.64, 0.525))
    _draw_box(ax, 0.82, 0.64, 0.11, 0.14, "DHW head", "Linear -> 1", face=ARCH_COLORS["head_reg"], title_size=10)
    _draw_box(ax, 0.82, 0.24, 0.11, 0.14, "Alert head", "Linear -> K", face=ARCH_COLORS["head_cls"], title_size=10)
    _draw_box(ax, 0.945, 0.64, 0.05, 0.14, "DHW", "h+28", face=ARCH_COLORS["output"], title_size=9.5)
    _draw_box(ax, 0.945, 0.24, 0.05, 0.14, "Alert", "h+28", face=ARCH_COLORS["output"], title_size=9.5)
    _arrow(ax, (0.77, 0.525), (0.82, 0.71), rad=0.18)
    _arrow(ax, (0.77, 0.525), (0.82, 0.31), rad=-0.18)
    _arrow(ax, (0.93, 0.71), (0.945, 0.71))
    _arrow(ax, (0.93, 0.31), (0.945, 0.31))
    _small_label(ax, 0.875, 0.50, "multi-task branching")
    plt.show()


def draw_stgnn_architecture(figsize=(14, 5.2)):
    fig, ax = _setup_arch_canvas(
        "ST-GNN: kết hợp không gian và thời gian",
        "Mỗi ô lưới là một node; GNN trao đổi tín hiệu giữa node lân cận trước khi encoder thời gian dự báo.",
        figsize=figsize,
    )
    node_xy = np.array([[0.07, 0.66], [0.13, 0.72], [0.18, 0.63], [0.08, 0.50], [0.15, 0.52], [0.20, 0.45]])
    edges = [(0, 1), (1, 2), (0, 3), (1, 4), (2, 4), (3, 4), (4, 5)]
    for i, j in edges:
        ax.plot([node_xy[i, 0], node_xy[j, 0]], [node_xy[i, 1], node_xy[j, 1]], color="#94a3b8", lw=1.8)
    for k, (x, y) in enumerate(node_xy):
        circle = Circle((x, y), 0.022, facecolor="#dbeafe", edgecolor=EDGE, lw=1.3)
        ax.add_patch(circle)
        ax.text(x, y, str(k), ha="center", va="center", fontsize=8.5, fontweight="bold")
    _small_label(ax, 0.135, 0.39, "reef grid graph")
    _draw_box(ax, 0.27, 0.56, 0.18, 0.17, "Input tensor", "batch x 90\nx node x feature", face=ARCH_COLORS["data"])
    _draw_box(ax, 0.50, 0.56, 0.18, 0.17, "Graph layer", "message passing\ngiữa node", face=ARCH_COLORS["encoder"])
    _draw_box(ax, 0.50, 0.25, 0.18, 0.17, "Temporal encoder", "GRU/CNN/attention\ntheo 90 ngày", face=ARCH_COLORS["feature"])
    _draw_box(ax, 0.72, 0.40, 0.13, 0.17, "Node hidden", "biểu diễn\ncho từng ô", face=ARCH_COLORS["hidden"])
    _draw_box(ax, 0.88, 0.61, 0.10, 0.13, "DHW/node", "regression", face=ARCH_COLORS["head_reg"], title_size=9.5)
    _draw_box(ax, 0.88, 0.23, 0.10, 0.13, "Alert/node", "classification", face=ARCH_COLORS["head_cls"], title_size=9.5)
    _arrow(ax, (0.22, 0.58), (0.27, 0.645))
    _arrow(ax, (0.45, 0.645), (0.50, 0.645))
    _arrow(ax, (0.59, 0.56), (0.59, 0.42))
    _arrow(ax, (0.68, 0.335), (0.72, 0.485))
    _arrow(ax, (0.85, 0.485), (0.88, 0.675), rad=0.18)
    _arrow(ax, (0.85, 0.485), (0.88, 0.295), rad=-0.18)
    _draw_box(ax, 0.27, 0.25, 0.18, 0.13, "Adjacency", "ai nối với ai,\ntrọng số bao nhiêu", face="#f8fafc", edge="#64748b", title_size=9.5)
    _arrow(ax, (0.36, 0.38), (0.52, 0.56), color="#64748b", rad=-0.20)
    plt.show()


draw_pipeline_architecture()

## 3. Feature engineering: đưa trí nhớ vào tabular model

Tabular model nhìn mỗi dòng như một vector feature độc lập. Nó không tự biết hôm qua, tuần trước hoặc tháng trước xảy ra gì. Vì vậy ta phải mã hóa trí nhớ bằng feature:

| Nhóm feature | Ý tưởng | Ví dụ |
|---|---|---|
| Lag | Giá trị ở một mốc quá khứ cụ thể | `CRW_DHW_lag7`, `CRW_HOTSPOT_lag28` |
| Rolling | Tóm tắt một cửa sổ quá khứ | `CRW_DHW_roll28_mean`, `CRW_HOTSPOT_roll14_max` |
| Calendar | Vị trí trong chu kỳ mùa vụ | `doy_sin`, `doy_cos`, `month_sin`, `month_cos` |

Thông điệp chính: với Random Forest, XGBoost và LightGBM, chất lượng "trí nhớ" nằm ở feature engineering, không nằm trong bản thân mô hình cây.

In [ ]:
metadata_cols = {"time", "target_time_h28", "node_id", "latitude", "longitude", "alert_label"}
target_cols = {TARGET_REG, TARGET_CLS}
feature_cols = [
    c for c in tabular.columns
    if c not in metadata_cols and c not in target_cols and not c.startswith("target_")
    and pd.api.types.is_numeric_dtype(tabular[c])
]

family_rows = []
for family, predicate in {
    "base/current": lambda c: c.startswith("CRW_") and "_lag" not in c and "_roll" not in c,
    "lag": lambda c: "_lag" in c,
    "rolling": lambda c: "_roll" in c,
    "calendar/cyclic": lambda c: c in {"doy_sin", "doy_cos", "month_sin", "month_cos", "month", "day_of_year"},
}.items():
    cols = [c for c in feature_cols if predicate(c)]
    family_rows.append({"feature_family": family, "n_columns": len(cols), "examples": ", ".join(cols[:5])})

display(pd.DataFrame(family_rows))

focus_cols = [
    "time", "CRW_DHW", "CRW_DHW_lag7", "CRW_DHW_lag28",
    "CRW_DHW_roll28_mean", "CRW_HOTSPOT_roll14_max", "doy_sin", "doy_cos",
    TARGET_REG, TARGET_CLS,
]
display(tabular[focus_cols].head(6))

## 4. Từ Decision Tree đến Random Forest, XGBoost và LightGBM

### Decision Tree

Decision Tree học các câu hỏi kiểu nếu-thì, ví dụ:

```text
Nếu CRW_DHW > 4
    Nếu CRW_HOTSPOT_roll14_max > 1 -> rủi ro cao
    Ngược lại                         -> rủi ro trung bình
Ngược lại
    Nếu CRW_BAA_7D_MAX_lag7 >= 3      -> cần chú ý
    Ngược lại                         -> rủi ro thấp
```

Cây mạnh ở các bài toán có ngưỡng và tương tác biến, nhưng một cây đơn lẻ dễ overfit.

### Random Forest

Random Forest xây nhiều cây tương đối độc lập bằng bootstrap sampling và random feature selection. Hồi quy lấy trung bình, phân loại lấy bỏ phiếu đa số.

### Boosting: XGBoost và LightGBM

Boosting xây cây theo chuỗi. Cây sau tập trung sửa lỗi còn lại của các cây trước. XGBoost nhấn mạnh regularization và kiểm soát overfit; LightGBM nhấn mạnh tốc độ và cách phát triển cây hiệu quả.

In [ ]:
tabular_architecture = pd.DataFrame([
    {
        "model": "Decision Tree",
        "architecture": "Một cây điều kiện nếu-thì",
        "learns": "Ngưỡng và tương tác feature",
        "main_risk": "Overfit nếu cây quá sâu",
    },
    {
        "model": "Random Forest",
        "architecture": "Nhiều cây độc lập, bootstrap + random features",
        "learns": "Nhiều góc nhìn ổn định trên feature lag/rolling",
        "main_risk": "Không tự hiểu thứ tự thời gian; cần feature tốt",
    },
    {
        "model": "XGBoost",
        "architecture": "Chuỗi cây, mỗi cây sửa phần lỗi còn lại",
        "learns": "Tương tác phức tạp, vùng chuyển tiếp gần ngưỡng",
        "main_risk": "Overfit nếu learning rate/depth/n_estimators không kiểm soát",
    },
    {
        "model": "LightGBM",
        "architecture": "Gradient boosting tối ưu tốc độ, thường leaf-wise",
        "learns": "Tổ hợp feature quan trọng trên bảng nhiều cột",
        "main_risk": "Cần validation và class weight cho lớp hiếm",
    },
])
display(tabular_architecture)

In [ ]:
draw_ensemble_architecture()

## 5. Vì sao deep learning không tự động tốt hơn mô hình cây?

Một thông điệp quan trọng của buổi học: **deep learning không thay thế pipeline đúng**.

- Nếu dữ liệu là bảng và feature engineering tốt, Random Forest/XGBoost/LightGBM thường rất mạnh.
- Deep learning có lợi thế khi ta muốn học trực tiếp từ chuỗi, giữ thứ tự ngày và nhận diện pattern động học khó tóm tắt bằng rolling mean.
- Nhưng deep learning cần nhiều dữ liệu hơn, nhạy với normalization, learning rate, batch size, kiến trúc và dễ overfit khi lớp event hiếm.

Cách tiếp cận nên dùng trong dự án: tabular models là baseline bắt buộc; deep models là nhánh mở rộng để học chuỗi trực tiếp.

## 6. Sequence window: biến chuỗi thành đầu vào cho deep learning

Mỗi mẫu deep learning là một cửa sổ 90 ngày trước ngày `t`:

```text
X_seq shape = samples x 90 days x features_per_day
y_reg       = DHW tại t + 28
y_cls       = alert level tại t + 28
```

Khác với rolling mean, sequence window giữ lại thứ tự: mô hình có thể phân biệt "cao ở đầu cửa sổ" và "cao dần ở cuối cửa sổ".

In [ ]:
from coral_bleaching_pipeline.data import make_sequence_arrays

sequence_features = [
    "CRW_SST", "CRW_SSTANOMALY", "CRW_HOTSPOT", "CRW_DHW", "CRW_BAA", "CRW_BAA_7D_MAX",
    "doy_sin", "doy_cos", "month_sin", "month_cos",
]

aggregate_seq = aggregate.dropna(subset=sequence_features + [TARGET_REG, TARGET_CLS]).copy()
aggregate_seq[TARGET_CLS] = aggregate_seq[TARGET_CLS].astype(int)

x_seq, y_reg, y_cls, seq_meta = make_sequence_arrays(
    aggregate_seq,
    feature_cols=sequence_features,
    target_regression=TARGET_REG,
    target_classification=TARGET_CLS,
    sequence_length=SEQUENCE_LENGTH,
    target_time_col="target_time_h28",
)

print(f"x_seq shape: {x_seq.shape} = samples x days x features")
print(f"y_reg shape: {y_reg.shape}")
print(f"y_cls shape: {y_cls.shape}")
display(seq_meta.head())

## 7. Kiến trúc multi-task: một encoder, hai đầu ra

Vì DHW và alert level đều phụ thuộc vào lịch sử stress nhiệt, ta có thể dùng chung một encoder để học biểu diễn, rồi tách ra hai head:

```text
Input sequence 90 ngày
        |
Shared encoder: LSTM / GRU / CNN-LSTM / TFT-lite / ST-GNN
        |
Hidden representation
        |--------------------|
Regression head          Classification head
DHW h+28                 Alert level h+28
```

Loss tổng quát:

```text
Total loss = alpha * regression_loss + beta * classification_loss
```

Nếu lớp Alert Level 1/2 quá hiếm, cần tăng vai trò của classification loss, class weight, focal loss hoặc threshold tuning.

In [ ]:
try:
    import torch
    from coral_bleaching_pipeline.models.deep import (
        LSTMForecastNet,
        GRUForecastNet,
        CNNLSTMForecastNet,
        TemporalFusionTransformerLite,
        SpatioTemporalGNN,
    )
    TORCH_AVAILABLE = True
except Exception as exc:
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

input_size = len(sequence_features)
num_classes = int(max(tabular[TARGET_CLS].max(), aggregate_seq[TARGET_CLS].max())) + 1

if TORCH_AVAILABLE:
    hidden_size = 32
    dropout = 0.10
    adjacency = np.load(ADJACENCY_PATH)
    deep_models = {
        "lstm": LSTMForecastNet(input_size, hidden_size, num_layers=1, dropout=dropout, num_classes=num_classes),
        "gru": GRUForecastNet(input_size, hidden_size, num_layers=1, dropout=dropout, num_classes=num_classes),
        "cnn_lstm": CNNLSTMForecastNet(input_size, hidden_size, num_layers=1, dropout=dropout, cnn_channels=24, num_classes=num_classes),
        "tft_lite": TemporalFusionTransformerLite(input_size, SEQUENCE_LENGTH, hidden_size=32, dropout=dropout, heads=4, layers=1, num_classes=num_classes),
        "st_gnn": SpatioTemporalGNN(input_size, adjacency=adjacency, hidden_size=24, dropout=dropout, num_classes=num_classes),
    }
    rows = []
    for name, model in deep_models.items():
        rows.append({"model": name, "trainable_params": count_params(model), "class_outputs": num_classes})
    display(pd.DataFrame(rows).sort_values("trainable_params"))
else:
    display(Markdown(f"Không import được PyTorch hoặc model deep: `{TORCH_IMPORT_ERROR}`"))

In [ ]:
if TORCH_AVAILABLE:
    torch.manual_seed(42)
    batch_size = 4
    x_dummy = torch.randn(batch_size, SEQUENCE_LENGTH, input_size)
    shape_rows = []
    for name, model in deep_models.items():
        model.eval()
        with torch.no_grad():
            if name == "st_gnn":
                n_nodes = adjacency.shape[0]
                x_graph = torch.randn(2, SEQUENCE_LENGTH, n_nodes, input_size)
                reg_out, cls_out = model(x_graph)
            else:
                reg_out, cls_out = model(x_dummy)
        shape_rows.append({
            "model": name,
            "regression_output_shape": tuple(reg_out.shape),
            "classification_output_shape": tuple(cls_out.shape),
        })
    display(pd.DataFrame(shape_rows))

## 8. Cách giảng từng kiến trúc deep model

| Mô hình | Kiến trúc lõi | Ý tưởng học | Khi nào nên thử | Rủi ro |
|---|---|---|---|---|
| LSTM | Input `90 x F` -> LSTM -> hidden cuối -> 2 heads | Bộ nhớ có cổng, giữ thông tin dài hạn | Khi stress tích lũy theo nhiều tuần | Dễ overfit nếu dữ liệu/event ít |
| GRU | Input `90 x F` -> GRU -> hidden cuối -> 2 heads | Bộ nhớ gọn hơn LSTM | Khi muốn baseline sequence nhẹ hơn | Có thể kém linh hoạt hơn LSTM |
| CNN-LSTM | Conv1D -> LSTM -> 2 heads | CNN phát hiện pattern ngắn, LSTM học diễn biến dài | Khi có dấu hiệu cục bộ như HotSpot tăng vài ngày | Thêm tham số, cần tuning kernel/filter |
| TFT-lite | Projection -> positional embedding -> transformer attention -> 2 heads | Attention chọn đoạn thời gian quan trọng | Khi quan hệ xa trong 90 ngày quan trọng | Không tự chứng minh nhân quả; cần dữ liệu đủ |
| ST-GNN | Node features -> message passing -> temporal encoder -> node outputs | Học cả quan hệ không gian và thời gian | Khi có nhiều ô lưới và lân cận có ý nghĩa | Phức tạp, lợi thế không rõ nếu rất ít node |

In [ ]:
architecture_specs = [
    {
        "title": "LSTM multi-task architecture",
        "note": "LSTM đọc tuần tự 90 ngày và giữ lại tín hiệu dài hạn như HotSpot kéo dài hoặc DHW tăng liên tục.",
        "blocks": [
            ("LSTM encoder", "forget/input/output gates", ARCH_COLORS["encoder"]),
            ("Last hidden", "tóm tắt chuỗi 90 ngày", ARCH_COLORS["feature"]),
            ("LayerNorm", "ổn định biểu diễn", ARCH_COLORS["hidden"]),
        ],
    },
    {
        "title": "GRU multi-task architecture",
        "note": "GRU là biến thể gọn hơn LSTM, phù hợp làm baseline sequence nhẹ và nhanh.",
        "blocks": [
            ("GRU encoder", "update/reset gates", ARCH_COLORS["encoder"]),
            ("Last hidden", "trí nhớ gọn của chuỗi", ARCH_COLORS["feature"]),
            ("LayerNorm", "giảm dao động scale", ARCH_COLORS["hidden"]),
        ],
    },
    {
        "title": "CNN-LSTM architecture",
        "note": "CNN đánh dấu pattern ngắn như HotSpot tăng vài ngày; LSTM đọc các pattern đó theo thời gian.",
        "blocks": [
            ("Conv1D", "kernel 3-7 ngày", ARCH_COLORS["encoder"]),
            ("BatchNorm + GELU", "làm sạch tín hiệu cục bộ", ARCH_COLORS["feature"]),
            ("LSTM encoder", "diễn biến dài hơn", ARCH_COLORS["encoder"]),
            ("Last hidden", "stress representation", ARCH_COLORS["hidden"]),
        ],
    },
    {
        "title": "TFT-lite / attention architecture",
        "note": "Attention học ngày hoặc đoạn nào trong 90 ngày đáng chú ý hơn cho dự báo h+28.",
        "blocks": [
            ("Projection", "F feature -> hidden", ARCH_COLORS["encoder"]),
            ("Position", "giữ vị trí ngày", ARCH_COLORS["feature"]),
            ("Transformer encoder", "temporal attention", ARCH_COLORS["encoder"]),
            ("Context vector", "tập trung đoạn quan trọng", ARCH_COLORS["hidden"]),
        ],
    },
]

for spec in architecture_specs:
    draw_multitask_architecture(spec["title"], spec["blocks"], spec["note"])

draw_stgnn_architecture()

## 9. ST-GNN: khi bài toán có cả không gian và thời gian

Trong dữ liệu quanh Lizard Island, mỗi ô lưới có thể xem là một node. Các node gần nhau được nối bằng cạnh thông qua adjacency matrix.

Ý tưởng kiến trúc:

```text
Node = ô lưới
Edge = quan hệ gần nhau hoặc ảnh hưởng không gian
Feature node = SST, anomaly, HotSpot, DHW, BAA, ...
Temporal window = 90 ngày
GNN = trao đổi thông tin giữa node lân cận
Temporal module = học diễn biến theo thời gian
Output = dự báo DHW/alert cho node hoặc vùng
```

Câu hỏi nên đặt cho lớp: nếu chỉ có vài node rất giống nhau, ST-GNN có đáng để dùng không? Đây là lúc nhấn mạnh trade-off giữa độ phức tạp và lợi ích mô hình.

In [ ]:
nodes = pd.read_csv(NODES_PATH)
adjacency = np.load(ADJACENCY_PATH)

display(nodes)

fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(adjacency, cmap="viridis")
ax.set_title("Adjacency matrix giữa các ô lưới")
ax.set_xlabel("node j")
ax.set_ylabel("node i")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()

## 10. Mất cân bằng lớp và vì sao RMSE tốt chưa chắc event recall tốt

Trong cảnh báo tẩy trắng, phần lớn ngày là No Stress hoặc Watch. Alert Level 1/2 thường ít hơn nhiều nhưng lại quan trọng nhất về quản lý rủi ro.

Vì vậy cần tách ba nhóm metric:

| Mục tiêu | Metric nên xem |
|---|---|
| Dự báo giá trị DHW | MAE, RMSE, R2 |
| Phân loại tổng thể | accuracy, balanced accuracy, macro-F1 |
| Bắt cảnh báo cao | event precision, event recall, event F1 |

Câu chốt để giảng: một mô hình có thể sai trung bình ít nhưng vẫn bỏ sót những ngày nguy hiểm nếu nó đánh giá thấp vùng gần ngưỡng Alert Level 1/2.

In [ ]:
def alert_from_dhw(dhw):
    dhw = np.asarray(dhw)
    alert = np.zeros_like(dhw, dtype=int)
    alert[(dhw >= 0.1) & (dhw < 4.0)] = 2
    alert[(dhw >= 4.0) & (dhw < 8.0)] = 3
    alert[dhw >= 8.0] = 4
    return alert

def event_recall(y_true_alert, y_pred_alert, threshold=3):
    true_event = y_true_alert >= threshold
    pred_event = y_pred_alert >= threshold
    return (true_event & pred_event).sum() / max(true_event.sum(), 1)

examples = pd.DataFrame({
    "case": ["near Alert Level 1", "near Alert Level 2", "low stress"],
    "true_dhw": [4.1, 8.0, 0.6],
    "pred_dhw": [3.8, 6.5, 0.4],
})
examples["absolute_error"] = (examples["true_dhw"] - examples["pred_dhw"]).abs()
examples["true_alert"] = alert_from_dhw(examples["true_dhw"])
examples["pred_alert"] = alert_from_dhw(examples["pred_dhw"])
examples["event_missed"] = (examples["true_alert"] >= 3) & (examples["pred_alert"] < 3)
display(examples)

rmse = np.sqrt(np.mean((examples["true_dhw"] - examples["pred_dhw"]) ** 2))
recall = event_recall(examples["true_alert"], examples["pred_alert"])
print(f"RMSE demo: {rmse:.3f}")
print(f"Event recall demo: {recall:.3f}")

## Hoạt động trên lớp

### Hoạt động 1: kiểm tra leakage

Hỏi người học: feature nào dưới đây bị leakage?

- `CRW_DHW_lag7`
- `CRW_HOTSPOT_roll28_mean` nếu rolling chỉ dùng dữ liệu trước ngày `t`
- `CRW_DHW_future7_mean`
- `target_CRW_DHW_h28`

Đáp án cần nhấn mạnh: bất kỳ feature nào dùng tương lai hoặc dùng trực tiếp target đều làm kết quả đánh giá đẹp giả tạo.

### Hoạt động 2: chọn kiến trúc

Cho ba tình huống và yêu cầu chọn mô hình:

| Tình huống | Gợi ý mô hình | Lý do |
|---|---|---|
| Feature lag/rolling đã rất tốt, cần baseline mạnh | XGBoost/LightGBM | Mạnh với tabular và train nhanh |
| Muốn học trực tiếp chuỗi 90 ngày | LSTM hoặc GRU | Giữ thứ tự thời gian |
| Dấu hiệu cảnh báo là các đoạn HotSpot tăng ngắn trước khi DHW tăng | CNN-LSTM | CNN phát hiện pattern cục bộ |
| Có nhiều node không gian và muốn dùng thông tin lân cận | ST-GNN | Message passing giữa ô lưới |

### Hoạt động 3: vẽ kiến trúc multi-task

Yêu cầu người học vẽ lại kiến trúc chung:

```text
Input -> Encoder chung -> Hidden representation -> Regression head + Classification head
```

Sau đó hỏi: nếu mô hình dự báo DHW tốt nhưng event recall thấp, nên chỉnh ở feature, loss, threshold hay sampling?

## Chốt buổi

Ba ý cần người học mang về:

1. **Pipeline quyết định tính đúng đắn của bài toán**: horizon, leakage và time split phải đúng trước khi bàn đến model.
2. **Kiến trúc phải khớp với cách biểu diễn dữ liệu**: tabular model cần lag/rolling, sequence model cần cửa sổ `90 x F`, ST-GNN cần node/edge.
3. **Metric phải khớp với mục tiêu quản lý rủi ro**: RMSE tốt chưa đủ nếu mô hình bỏ sót Alert Level 1/2.

Gợi ý cho buổi thực hành tiếp theo: chọn một baseline tabular và một sequence model, chạy cùng train/validation/test split, sau đó so sánh RMSE, macro-F1 và event recall.